# Tokenizer Fertility and RankMe: Does Tokenization Efficiency Shape Representation Geometry?

**Author:** Giacomo Porpiglia  
**Model:** Apertus-8B (`swiss-ai/Apertus-8B-2509`)  
**Dataset:** Wikipedia (`wikimedia/wikipedia`, 13 languages)

---

## Research Question

> **Does tokenizer *fertility* — the average number of subword tokens generated per word by the Apertus tokenizer — correlate with the RankMe score of the resulting representations? And how does this relationship evolve across layers and training checkpoints?**

Tokenizer **fertility** for language $\ell$ is defined as:

$$\text{fertility}(\ell) = \frac{\sum_{t \in \mathcal{T}_\ell} |\text{tokenize}(t)|}{\sum_{t \in \mathcal{T}_\ell} |\text{words}(t)|}$$

For script-continuous languages (Chinese, Japanese) where word boundaries are absent, $|\text{words}(t)|$ is replaced by $|\text{chars}(t)|$ (non-whitespace characters), so the unit becomes *tokens per character*.

A high-fertility language is one the tokenizer encodes **inefficiently** — it fragments text into many small subword pieces rather than whole morphemes. We investigate whether this fragmentation leaves a measurable signature in the *geometry* of learned representations, as captured by **RankMe** (the effective dimensionality of the activation matrix, measured via Shannon entropy of singular values).

### Hypotheses

| # | Hypothesis | Why it might hold |
|---|------------|-------------------|
| H1 | **Layer-wise signature**: fertility predicts RankMe at the final checkpoint, especially in late layers. | Semantic information is concentrated in later layers; poor tokenization may reduce representational diversity there. |
| H2 | **Depth profile**: the fertility–RankMe correlation changes across depth. | Early layers capture token-level patterns (directly tied to tokenization); later layers capture abstractions. |
| H3 | **Training trajectory**: high- and low-fertility languages follow different RankMe growth curves. | The model may learn efficient representations for low-fertility languages faster/more completely. |


In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

## 0. Load Results

Primary source: `results/apertus_wiki_all.csv` (Wikipedia run).  
Fallback: `results/apertus.csv` (FineWeb2 run) — used for development until the wiki run is complete.

The full wiki pipeline is defined in `geometry_analysis/configs/apertus_wiki.yaml`:
```
python geometry_analysis/geometry_analysis.py --config geometry_analysis/configs/apertus_wiki.yaml
```

In [ ]:
WIKI_CSV = "../results/apertus_wiki_all.csv"
FW2_CSV  = "../results/apertus.csv"

if os.path.exists(WIKI_CSV):
    df = pd.read_csv(WIKI_CSV)
    data_label = "Wikipedia"
else:
    print(f"[NOTE] {WIKI_CSV} not found — using FineWeb2 results as preview.")
    df = pd.read_csv(FW2_CSV)
    data_label = "FineWeb2 (preview)"

print(f"Loaded {len(df):,} rows | source: {data_label}")
print(f"Checkpoints : {df.checkpoint.nunique()}")
print(f"Languages   : {sorted(df.dataset.unique())}")
print(f"Layers      : {df.layer.nunique()}")
print(f"Aggregations: {df.aggregation.unique().tolist()}")
df.head(3)

In [ ]:
def ckpt_key(name):
    """Numeric sort key for checkpoint names: tokens seen, 'main' goes last."""
    if str(name).lower() == "main":
        return float("inf")
    m = re.match(r"step\d+-tokens(\d+)([BT])", str(name), re.IGNORECASE)
    if m:
        val = float(m.group(1))
        return val * 1000 if m.group(2).upper() == "T" else val
    return float("inf") - 1

def layer_num(name):
    m = re.search(r"(\d+)", str(name))
    return int(m.group(1)) if m else 0

def ckpt_label(name):
    """Compact checkpoint label: '210B', '15T', 'main'."""
    if str(name).lower() == "main":
        return "main"
    m = re.match(r"step\d+-tokens(\d+)([BT])", str(name), re.IGNORECASE)
    return f"{m.group(1)}{m.group(2)}" if m else name

checkpoints  = sorted(df.checkpoint.unique(), key=ckpt_key)
languages    = sorted(df.dataset.unique())
layers       = sorted(df.layer.unique(), key=layer_num)
layer_nums   = [layer_num(l) for l in layers]
ckpt_labels  = [ckpt_label(c) for c in checkpoints]

# Use last-token aggregation throughout (causal LM prediction target)
df_last = df[df.aggregation == "last"].copy()

print(f"Checkpoints : {checkpoints[0]} … {checkpoints[-1]}  ({len(checkpoints)} total)")
print(f"Layers      : layer_{layer_nums[0]} … layer_{layer_nums[-1]}  ({len(layers)} total)")

## 1. Tokenizer Fertility

We load the Apertus tokenizer and sample Wikipedia articles for each language to measure how many subword tokens are generated per word (or per character for Chinese and Japanese).

**Fallback values** (approximate, based on published Llama-3 tokenizer benchmarks) are pre-loaded below. Run the computation cell on the cluster to replace them with exact values for the Apertus tokenizer.

> **Why Wikipedia?** Wikipedia provides encyclopedic, formally written text across all 13 languages. Using the same domain for both fertility computation and RankMe analysis (wiki config) ensures the measurements are directly comparable.

In [ ]:
# Approximate fertility values — replace with computed values by running the next cell.
# Unit: tokens/word for word-boundary languages; tokens/char for Chinese and Japanese.
APPROX_FERTILITY = {
    "English":    (1.32, "tok/word"),
    "Spanish":    (1.46, "tok/word"),
    "Italian":    (1.47, "tok/word"),
    "French":     (1.55, "tok/word"),
    "Indonesian": (1.67, "tok/word"),
    "German":     (1.87, "tok/word"),
    "Swahili":    (1.84, "tok/word"),
    "Turkish":    (2.18, "tok/word"),
    "Vietnamese": (2.32, "tok/word"),
    "Arabic":     (2.96, "tok/word"),
    "Hindi":      (4.73, "tok/word"),
    "Chinese":    (1.52, "tok/char"),
    "Japanese":   (1.87, "tok/char"),
}

fertility = {lang: v  for lang, (v, _) in APPROX_FERTILITY.items()}
fert_unit = {lang: u  for lang, (_, u) in APPROX_FERTILITY.items()}
print("Fallback fertility values loaded.")
print("Override by setting COMPUTE=True in the next cell and running on the cluster.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  CLUSTER CELL — set COMPUTE = True to replace approximate values above.
#  Requires: transformers, datasets, internet access, HuggingFace token if needed.
# ─────────────────────────────────────────────────────────────────────────────
COMPUTE = False

if COMPUTE:
    from transformers import AutoTokenizer
    from datasets import load_dataset
    import json, pathlib

    TOKENIZER_NAME = "swiss-ai/Apertus-8B-2509"
    N_ARTICLES = 300    # Wikipedia articles sampled per language
    CJK = {"Chinese", "Japanese"}

    WIKI_SUBSETS = {
        "English":    "20231101.en",  "Chinese":    "20231101.zh",
        "French":     "20231101.fr",  "Indonesian": "20231101.id",
        "Italian":    "20231101.it",  "Swahili":    "20231101.sw",
        "Spanish":    "20231101.es",  "German":     "20231101.de",
        "Hindi":      "20231101.hi",  "Arabic":     "20231101.ar",
        "Turkish":    "20231101.tr",  "Vietnamese": "20231101.vi",
        "Japanese":   "20231101.ja",
    }

    def _compute_fertility(tokenizer, texts, lang):
        tok_total, unit_total = 0, 0
        for text in texts:
            text = text.strip()
            if not text:
                continue
            ids = tokenizer.encode(text, add_special_tokens=False)
            tok_total += len(ids)
            if lang in CJK:
                unit_total += len("".join(text.split()))   # non-whitespace chars
            else:
                unit_total += len(text.split())             # whitespace-split words
        return tok_total / unit_total if unit_total else float("nan")

    print(f"Loading tokenizer: {TOKENIZER_NAME}")
    tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME, trust_remote_code=True)
    print(f"Vocab size: {tok.vocab_size:,}\n")

    computed_fertility, computed_unit = {}, {}
    for lang, subset in WIKI_SUBSETS.items():
        print(f"  {lang}...", end=" ", flush=True)
        ds = load_dataset("wikimedia/wikipedia", subset, split="train", streaming=True)
        texts = [item["text"] for _, item in zip(range(N_ARTICLES), ds)]
        f = _compute_fertility(tok, texts, lang)
        unit = "tok/char" if lang in CJK else "tok/word"
        computed_fertility[lang] = round(f, 4)
        computed_unit[lang] = unit
        print(f"{f:.3f} {unit}")

    # Persist results for reproducibility
    out_path = pathlib.Path("../results/fertility_apertus_wiki.json")
    out_path.write_text(json.dumps({"fertility": computed_fertility, "unit": computed_unit}, indent=2))
    print(f"\nSaved to {out_path}")

    fertility = computed_fertility
    fert_unit = computed_unit
    print("Fertility values updated from tokenizer computation.")
else:
    print("COMPUTE=False — using approximate fallback values.")

In [ ]:
df_fert = (
    pd.DataFrame({
        "language":  list(fertility.keys()),
        "fertility": list(fertility.values()),
        "unit":      [fert_unit[l] for l in fertility],
    })
    .sort_values("fertility")
    .reset_index(drop=True)
)

# Languages available in the RankMe results
common_langs = [l for l in df_fert.language if l in languages]

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ["#c0392b" if fert_unit[lang] == "tok/char" else "#2980b9"
              for lang in df_fert.language]
ax.barh(df_fert.language, df_fert.fertility, color=bar_colors, height=0.7, edgecolor="white")
ax.axvline(df_fert.fertility.mean(), ls="--", color="gray", lw=1.5,
           label=f"Mean = {df_fert.fertility.mean():.2f}")
ax.set_xlabel("Fertility", fontsize=12)
ax.set_title("Tokenizer Fertility by Language (Apertus tokenizer, Wikipedia samples)", fontsize=13)
legend_elems = [
    Patch(facecolor="#2980b9", label="tokens / word  (word-boundary languages)"),
    Patch(facecolor="#c0392b", label="tokens / char  (Chinese, Japanese — no word boundaries)"),
]
ax.legend(handles=legend_elems, loc="lower right", fontsize=9)
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.savefig("fertility_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print(df_fert.to_string(index=False))

## 2. RankMe at the Final Checkpoint

We examine the RankMe layer profile for each language at the final training checkpoint (`main`), using the **last-token** aggregation.

Lines are colored by fertility (low = bright, high = dark), so visual clusters immediately reveal whether fertility tracks RankMe level.

In [ ]:
FINAL_CKPT = "main" if "main" in checkpoints else checkpoints[-1]
df_final = df_last[df_last.checkpoint == FINAL_CKPT]

fert_vals = [fertility[l] for l in common_langs]
fmin, fmax = min(fert_vals), max(fert_vals)
cmap = plt.cm.plasma_r   # low fertility = bright/yellow, high = dark/purple

fig, ax = plt.subplots(figsize=(12, 5))
for lang in sorted(common_langs, key=lambda l: fertility[l]):
    sub = (df_final[df_final.dataset == lang]
           .sort_values("layer", key=lambda s: s.map(layer_num)))
    if sub.empty:
        continue
    f = fertility[lang]
    color = cmap((f - fmin) / (fmax - fmin + 1e-9))
    ax.plot(
        sub.layer.map(layer_num), sub.rankme,
        marker="o", ms=4, lw=1.8, color=color,
        label=f"{lang}  [{f:.2f} {fert_unit.get(lang, '')}]",
    )

ax.set_xlabel("Layer")
ax.set_ylabel("RankMe")
ax.set_title(
    f"RankMe Layer Profiles — checkpoint: {FINAL_CKPT}  |  last-token  |  {data_label}\n"
    "Color gradient: low fertility (bright) → high fertility (dark)",
    fontsize=12,
)
ax.legend(title="Language (fertility)", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("layer_profiles_final.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Fertility–RankMe Correlation

For each layer at the final checkpoint, we compute Pearson $r$ and Spearman $\rho$ between fertility and RankMe across the 13 languages.

> **Caveat:** With only 13 data points the correlation estimates carry wide confidence intervals. The analysis should be read as exploratory — we look for consistent directional trends across layers rather than individually significant p-values.

### 3a. Scatter Plots at Selected Layers

In [ ]:
n = len(layers)
sel_layers = [layers[0], layers[n // 4], layers[n // 2], layers[3 * n // 4], layers[-1]]

fig, axes = plt.subplots(1, len(sel_layers), figsize=(16, 4), sharey=False)
for ax, layer in zip(axes, sel_layers):
    sub = df_final[df_final.layer == layer]
    xs, ys, labels = [], [], []
    for lang in common_langs:
        row = sub[sub.dataset == lang]
        if not row.empty:
            xs.append(fertility[lang])
            ys.append(row.rankme.values[0])
            labels.append(lang)

    xs, ys = np.array(xs), np.array(ys)
    if len(xs) < 3:
        ax.set_title(f"L{layer_num(layer)}\n(no data)")
        continue

    r,   p   = pearsonr(xs, ys)
    rho, _   = spearmanr(xs, ys)

    ax.scatter(xs, ys, s=65, color="#2c3e50", zorder=3)
    for x, y, lab in zip(xs, ys, labels):
        ax.annotate(lab[:3], (x, y), xytext=(4, 2), textcoords="offset points", fontsize=7)

    z    = np.polyfit(xs, ys, 1)
    xfit = np.linspace(xs.min(), xs.max(), 80)
    ax.plot(xfit, np.polyval(z, xfit), "r--", lw=1.5, alpha=0.8)

    ax.set_title(f"Layer {layer_num(layer)}\nr={r:.2f}  ρ={rho:.2f}\np={p:.2f}", fontsize=9)
    ax.set_xlabel("Fertility")
    if layer == sel_layers[0]:
        ax.set_ylabel("RankMe")
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Fertility vs. RankMe at Selected Layers — {FINAL_CKPT}  |  {data_label}",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.savefig("scatter_fertility_rankme.png", dpi=150, bbox_inches="tight")
plt.show()

### 3b. Correlation Profile Across All Layers

The profile shows how tightly fertility and RankMe track each other as a function of depth.  
Gold bands mark layers where Pearson $r$ is statistically significant at $p < 0.05$.

In [ ]:
prs, rhos, pvals = [], [], []
for layer in layers:
    sub = df_final[df_final.layer == layer]
    xs, ys = [], []
    for lang in common_langs:
        row = sub[sub.dataset == lang]
        if not row.empty:
            xs.append(fertility[lang])
            ys.append(row.rankme.values[0])
    if len(xs) < 3:
        prs.append(np.nan); rhos.append(np.nan); pvals.append(np.nan)
        continue
    r, p   = pearsonr(xs, ys)
    rho, _ = spearmanr(xs, ys)
    prs.append(r); rhos.append(rho); pvals.append(p)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(layer_nums, prs,  "o-",  color="#2980b9", lw=2, ms=5, label="Pearson r")
ax.plot(layer_nums, rhos, "s--", color="#e74c3c", lw=2, ms=5, label="Spearman ρ")
ax.axhline(0, color="gray", lw=1, ls=":")
ax.fill_between(layer_nums, prs, 0, alpha=0.1, color="#2980b9")

sig_added = False
for lnum, p in zip(layer_nums, pvals):
    if p is not None and not np.isnan(p) and p < 0.05:
        label = "p < 0.05" if not sig_added else None
        ax.axvspan(lnum - 0.45, lnum + 0.45, alpha=0.18, color="gold", zorder=0, label=label)
        sig_added = True

ax.set_xlabel("Layer")
ax.set_ylabel("Correlation (fertility vs. RankMe)")
ax.set_title(
    f"Fertility–RankMe Correlation Profile Across Layers\n"
    f"{FINAL_CKPT}  |  last-token  |  {data_label}",
    fontsize=12,
)
ax.legend()
ax.set_xlim(-0.5, layer_nums[-1] + 0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("correlation_profile.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
df_corr = pd.DataFrame({"layer": layer_nums, "pearson_r": prs, "spearman_rho": rhos, "p_value": pvals})
print("Layers with |r| > 0.5:")
print(df_corr[df_corr.pearson_r.abs() > 0.5].to_string(index=False))

## 4. Training Dynamics by Fertility Tier

We split the 13 languages into three equal-sized fertility tiers (low / medium / high) and track mean RankMe over training at the **middle layer** — the depth where the most active learning of linguistic structure typically occurs.

A widening gap between tiers during training would confirm H3: the model learns more effective representations for low-fertility languages more quickly.

In [ ]:
df_fert_sorted = df_fert[df_fert.language.isin(common_langs)].sort_values("fertility").reset_index(drop=True)
k = max(1, len(df_fert_sorted) // 3)
low_tier  = df_fert_sorted.iloc[:k].language.tolist()
mid_tier  = df_fert_sorted.iloc[k:2*k].language.tolist()
high_tier = df_fert_sorted.iloc[2*k:].language.tolist()

print("Low  fertility:", low_tier)
print("Mid  fertility:", mid_tier)
print("High fertility:", high_tier)

TIERS = {
    "Low":    (low_tier,  "#2980b9"),
    "Medium": (mid_tier,  "#27ae60"),
    "High":   (high_tier, "#e74c3c"),
}
mid_layer = layers[len(layers) // 2]
df_mid = df_last[df_last.layer == mid_layer]

fig, ax = plt.subplots(figsize=(14, 5))
for tier, (langs, color) in TIERS.items():
    records = []
    for ci, ckpt in enumerate(checkpoints):
        sub = df_mid[(df_mid.checkpoint == ckpt) & (df_mid.dataset.isin(langs))]
        if not sub.empty:
            records.append({"idx": ci, "mean": sub.rankme.mean(), "std": sub.rankme.std()})
    if not records:
        continue
    tr = pd.DataFrame(records).sort_values("idx")
    xs = tr.idx.values
    ys = tr["mean"].values
    ax.plot(xs, ys, "o-", color=color, lw=2, ms=4,
            label=f"{tier} fertility ({', '.join(langs)})")
    ax.fill_between(xs, ys - tr["std"].values, ys + tr["std"].values, alpha=0.12, color=color)

ax.set_xticks(range(len(checkpoints)))
ax.set_xticklabels(ckpt_labels, rotation=45, ha="right", fontsize=7)
ax.set_xlabel("Checkpoint (tokens seen)")
ax.set_ylabel("RankMe (mean ± std within tier)")
ax.set_title(
    f"RankMe Training Dynamics by Fertility Tier\n"
    f"layer {layer_num(mid_layer)} | last-token | {data_label}",
    fontsize=12,
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_dynamics_tiers.png", dpi=150, bbox_inches="tight")
plt.show()

### 4b. Spearman ρ Heatmap — All Checkpoints × All Layers

Each cell shows the Spearman rank correlation between fertility and RankMe across the 13 languages for a given (checkpoint, layer) pair.

- **Blue**: higher fertility → higher RankMe (model compensates with richer representations)  
- **Red**: higher fertility → lower RankMe (fragmented tokens lead to more degenerate geometry)  
- **White**: no correlation

This view reveals *when* and *where* in the network the fertility signal emerges during training.

In [ ]:
rho_mat = np.full((len(checkpoints), len(layers)), np.nan)
for ci, ckpt in enumerate(checkpoints):
    for li, layer in enumerate(layers):
        sub = df_last[(df_last.checkpoint == ckpt) & (df_last.layer == layer)]
        xs, ys = [], []
        for lang in common_langs:
            row = sub[sub.dataset == lang]
            if not row.empty:
                xs.append(fertility[lang])
                ys.append(row.rankme.values[0])
        if len(xs) >= 3:
            rho, _ = spearmanr(xs, ys)
            rho_mat[ci, li] = rho

# Tick positions: every 4th layer
xtick_pos   = list(range(0, len(layers), 4))
xtick_labs  = [layer_nums[i] for i in xtick_pos]

fig, ax = plt.subplots(figsize=(14, 9))
im = ax.imshow(rho_mat, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
plt.colorbar(im, ax=ax, label="Spearman ρ  (fertility vs. RankMe)")
ax.set_xticks(xtick_pos)
ax.set_xticklabels(xtick_labs, fontsize=9)
ax.set_yticks(range(len(checkpoints)))
ax.set_yticklabels(ckpt_labels, fontsize=7)
ax.set_xlabel("Layer")
ax.set_ylabel("Checkpoint (tokens seen)")
ax.set_title(
    "Spearman ρ  (Fertility vs. RankMe)  Across Training and Layers\n"
    "Blue: higher fertility → higher RankMe  |  Red: higher fertility → lower RankMe",
    fontsize=12,
)
plt.tight_layout()
plt.savefig("spearman_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

mean_rho = np.nanmean(rho_mat)
print(f"Mean Spearman ρ across all (checkpoint, layer) pairs: {mean_rho:.3f}")
print(f"  > 0 (blue-dominant): fertility tends to INCREASE RankMe" if mean_rho > 0 else
      f"  < 0 (red-dominant) : fertility tends to DECREASE RankMe")

## 5. Conclusions

> **Run all cells on the cluster with `COMPUTE=True` and the wiki results to obtain exact findings. The template below lists what to look for and how to interpret each result.**

### Fertility Ranking (Section 1)

The approximate values suggest a clear split: **European languages** (English, Spanish, Italian, French) have low fertility (~1.3–1.6 tok/word), while **non-Latin-script languages** (Arabic, Hindi, Chinese, Japanese) have high fertility. This reflects the skew of the tokenizer's training corpus toward English-dominated data — a well-known property of Llama-style BPE tokenizers.

### Layer-wise Signature (H1, Section 2–3)

The layer profile plot (Section 2) shows whether high-fertility languages cluster above or below low-fertility ones. The correlation profile (Section 3b) quantifies this: if Pearson $r$ is consistently negative (red) in later layers, higher fertility systematically lowers RankMe — the tokenizer's fragmentation leads to more anisotropic, less diverse representations at depth.

### Depth Profile (H2, Section 3b)

Look for a change in $r$ as layer depth increases. A common pattern in multilingual LLMs is that early layers capture tokenization-level structure (where fertility effects may be strong) and later layers capture cross-lingual semantics (where fertility effects may weaken if the model has learned to abstract over tokenization).

### Training Dynamics (H3, Section 4)

The tier plot (Section 4a) shows whether fertility tiers **diverge** (model learns better representations for low-fertility languages faster), **converge** (model equalizes over training), or **remain parallel** (fertility is a fixed offset). The heatmap (Section 4b) shows when the fertility–RankMe relationship first emerges during training — a very early signal would suggest it is driven primarily by token frequency statistics, while a late-emerging signal would suggest it reflects learned semantic structure.

### Overall Interpretation

Tokenizer fertility is a proxy for a language's representation in the tokenizer's training data. Languages with poor tokenizer coverage (high fertility) fragment text into lower-information-density tokens. The model faces two possible regimes:

1. **Collapse regime** (fertility → ↓ RankMe): Fragmented tokens are highly predictable from neighbours (the next character of a word is almost determined by the current one), so the model learns redundant representations — lower RankMe.
2. **Richness regime** (fertility → ↑ RankMe): To handle the unpredictability of highly fragmented sequences the model maintains richer intermediate states — higher RankMe.

Which regime dominates, and at which layer, is the core empirical finding of this notebook.
